## 2.3. SVD (Singular Value Decomposition)

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

# --- Adım 0: Kurulum ve Örnek Metin ---
print("--- Adım 0: Kurulum ve Örnek Metin ---")
# Anlamlı ilişkiler içeren metnimizi tekrar kullanalım.
corpus_text = "kral, güçlü bir adamdır. kraliçe, bilge bir kadındır. adam, kral ile yaşar. kadın, kraliçe ile yaşar."
corpus = corpus_text.lower().replace('.', '').replace(',', '').split()

# Kelime dağarcığını oluştur
vocab = sorted(list(set(corpus)))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Metin Korpusu: {corpus}\n")
print(f"Kelime Dağarcığı (Vocabulary): {vocab}\n")

# Bağlam penceresi boyutu
window_size = 2
print(f"Bağlam Penceresi Boyutu: {window_size} (bir kelimenin sağından ve solundan {window_size} kelime alınacak)\n")
print("="*60)


# --- Adım 1: Birlikte Geçme Matrisini (Co-occurrence Matrix) Oluşturma ---
print("\n--- Adım 1: Birlikte Geçme Matrisini Oluşturma ---")
print("Bu adımda, hangi kelimenin hangi kelimeyle aynı bağlamda ne kadar sık geçtiğini sayıyoruz.")

# Başlangıçta tüm değerleri sıfır olan, (vocab_size x vocab_size) boyutunda bir matris oluştur
co_occurrence_matrix = np.zeros((vocab_size, vocab_size))

# Metin üzerinde pencereyi kaydırarak matrisi doldur
for i, center_word in enumerate(corpus):
    center_word_ix = word_to_ix[center_word]
    
    # Pencerenin başlangıç ve bitişini belirle
    start = max(0, i - window_size)
    end = min(len(corpus), i + window_size + 1)
    
    for j in range(start, end):
        # Merkez kelimenin kendisiyle olan ilişkisini sayma
        if i == j:
            continue
        context_word = corpus[j]
        context_word_ix = word_to_ix[context_word]
        co_occurrence_matrix[center_word_ix, context_word_ix] += 1

print("\nHam Birlikte Geçme Matrisi (İlk 5x5'lik kısım):")
print("Her hücre (A, B), A kelimesinin bağlamında B kelimesinin kaç kez geçtiğini gösterir.")
print(pd.DataFrame(co_occurrence_matrix, index=vocab, columns=vocab).iloc[:5, :5].to_string())
print("\nBu matris büyük, seyrek ve ham sayımlar anlamsal olarak çok anlamlı değil.")
print("="*60)


# --- Adım 2: PPMI Matrisine Dönüştürme (Daha Anlamlı İstatistikler) ---
print("\n--- Adım 2: Matrisi PPMI'a (Positive Pointwise Mutual Information) Dönüştürme ---")
print("PPMI, iki kelimenin şans eseri mi yoksa gerçekten bir ilişki içinde mi birlikte geçtiğini ölçer.")
print("Bu, 'bir', 'ile' gibi sık geçen ama anlamsız kelimelerin etkisini azaltır.\n")

# Bu adım genellikle daha iyi sonuçlar verir ancak konsepti basit tutmak için
# bu simülasyonda doğrudan ham matris üzerinden SVD uygulayacağız.
# Gerçek bir uygulamada bu adım kritik olabilir. PPMI'ın varlığını bilmek önemlidir.
print("Not: Bu simülasyonda, konsepti net tutmak için bu adımı atlayıp doğrudan SVD uygulayacağız.\n")
# matrix_to_reduce = ppmi(co_occurrence_matrix) # Gerçekte böyle bir fonksiyon olurdu.
matrix_to_reduce = co_occurrence_matrix
print("="*60)


# --- Adım 3: SVD Uygulama ve Boyut Azaltma ---
print("\n--- Adım 3: SVD ile Boyut Azaltma ---")
print("SVD, devasa ve seyrek olan birlikte geçme matrisini, anlamsal özünü koruyarak")
print("küçük ve yoğun (dense) vektörlere dönüştürür.\n")

# Numpy'ın SVD fonksiyonunu kullan
U, s, Vt = np.linalg.svd(matrix_to_reduce)

print(f"SVD, matrisi 3 parçaya ayırdı: U, s, Vt")
print(f"U matrisinin boyutu: {U.shape}  (Bu bizim için kelime vektörlerini içerir)")
print(f"s vektörünün (Sigma) boyutu: {s.shape} (Boyutların önemini gösterir)")
print(f"Vt matrisinin boyutu: {Vt.shape}\n")

# Kelime vektörlerimizin nihai boyutunu seçelim
embedding_dim = 3
print(f"Vektörlerimizi {embedding_dim} boyuta indireceğiz.\n")

# U matrisinin ilk 'embedding_dim' kadar sütununu alarak kelime vektörlerimizi oluşturuyoruz.
word_vectors = U[:, :embedding_dim]

print(f"Nihai Kelime Vektörleri Matrisi (Her satır bir kelimeye ait {embedding_dim} boyutlu vektör):")
# Daha okunaklı olması için pandas DataFrame kullanalım
import pandas as pd
vector_df = pd.DataFrame(word_vectors, index=vocab, columns=[f'boyut_{i+1}' for i in range(embedding_dim)])
print(vector_df.to_string())
print("\nArtık her kelimenin düşük boyutlu, yoğun bir vektör temsili var!")
print("="*60)


# --- Adım 4: Sorgulama ve Benzerlik Bulma ---
print("\n--- Adım 4: Bir Kelimeyi Sorgulama ve Anlamsal Benzerlik Bulma ---")

def cosine_similarity(vec1, vec2):
    """İki vektör arasındaki kosinüs benzerliğini hesaplar."""
    dot_product = np.dot(vec1, vec2)
    norm_prod = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return dot_product / norm_prod

def find_most_similar(word, embeddings, word_to_ix, ix_to_word):
    """Bir kelimeye en benzer kelimeleri bulur."""
    if word not in word_to_ix:
        print(f"'{word}' kelimesi kelime dağarcığında yok.")
        return

    query_ix = word_to_ix[word]
    query_vector = embeddings[query_ix]
    
    similarities = {}
    for i, vector in enumerate(embeddings):
        if i == query_ix:
            continue
        sim = cosine_similarity(query_vector, vector)
        similarities[ix_to_word[i]] = sim
        
    # Benzerlik skoruna göre sırala
    sorted_similarities = sorted(similarities.items(), key=lambda item: item[1], reverse=True)
    return sorted_similarities

# Örnek bir sorgu yapalım
query_word = 'kral'
print(f"Sorgu Kelimesi: '{query_word}'\n")

similar_words = find_most_similar(query_word, word_vectors, word_to_ix, ix_to_word)

print(f"'{query_word}' kelimesine en benzer kelimeler (benzerlik skoruna göre):")
if similar_words:
    for word, score in similar_words:
        print(f"- {word:<10} | Benzerlik: {score:.4f}")

--- Adım 0: Kurulum ve Örnek Metin ---
Metin Korpusu: ['kral', 'güçlü', 'bir', 'adamdır', 'kraliçe', 'bilge', 'bir', 'kadındır', 'adam', 'kral', 'ile', 'yaşar', 'kadın', 'kraliçe', 'ile', 'yaşar']

Kelime Dağarcığı (Vocabulary): ['adam', 'adamdır', 'bilge', 'bir', 'güçlü', 'ile', 'kadın', 'kadındır', 'kral', 'kraliçe', 'yaşar']

Bağlam Penceresi Boyutu: 2 (bir kelimenin sağından ve solundan 2 kelime alınacak)


--- Adım 1: Birlikte Geçme Matrisini Oluşturma ---
Bu adımda, hangi kelimenin hangi kelimeyle aynı bağlamda ne kadar sık geçtiğini sayıyoruz.

Ham Birlikte Geçme Matrisi (İlk 5x5'lik kısım):
Her hücre (A, B), A kelimesinin bağlamında B kelimesinin kaç kez geçtiğini gösterir.
         adam  adamdır  bilge  bir  güçlü
adam      0.0      0.0    0.0  1.0    0.0
adamdır   0.0      0.0    1.0  1.0    1.0
bilge     0.0      1.0    0.0  1.0    0.0
bir       1.0      1.0    1.0  0.0    1.0
güçlü     0.0      1.0    0.0  1.0    0.0

Bu matris büyük, seyrek ve ham sayımlar anlamsal olarak 